# 11c — Quadratic-form machinery verification

Covers `chap11_qf1.m` through `qf4.m`: pedagogical identity checks of the
`quadratic_form`/`quadratic_form_risk`/`quadratic_form_bond_portfolio1`/
`quadratic_form_bond_portfolio2`/`bond_portfolio_metrics` functions this
chapter's optimizers are built on. **None of these functions are present
in the shipped archive** except the elementary `quadratic_form(x,Q,R,c) =
0.5x'Qx - x'R + c` (the minus sign on the linear term is easy to miss —
it's pinned down below by testing both signs against `qf1.m`'s own
`qf5a`/`qf5b` shift-identity check, which only holds with the minus
convention; the "+R" convention that first looks natural fails that
check outright). The rest are reconstructed here from how they're
*called* — each call site fixes their required input/output contract
exactly, and this notebook's whole purpose (mirroring the source's own
purpose) is to numerically verify that reconstruction against
hand-derived formulas, so a wrong reconstruction would show up
immediately as a nonzero "diff" column below. All four scripts are
self-contained (hardcoded/random data, no external files). `qf1.m` draws
unseeded MATLAB random data (no `rng(...)` call), so exact figures are
not reproducible either from MATLAB or into Python; a NumPy
`default_rng` seed is used purely for this notebook's own
reproducibility, and every identity is checked to (near-)zero
difference regardless of the actual random draw.

In [1]:
import numpy as np
import pandas as pd

def quadratic_form(x, Q, R, c):
    # 0.5 x'Qx - x'R + c. The minus sign on the linear term is confirmed by
    # testing both signs against qf1.m's qf5a/qf5b shift identity: only
    # this convention makes quadratic_form(x-y,Q,R,c) == quadratic_form(x,
    # Q, R+Qy, 0.5 y'Qy + y'R + c) hold for arbitrary x,y,Q,R,c.
    x = np.asarray(x, dtype=float)
    return 0.5 * x @ Q @ x - x @ R + c


def _sector_risk_matrices(sector, X, X_star):
    # Q,R,c such that quadratic_form(z,Q,R,c) == 0.5*sum_j(sector_agg(z*X)_j - X_star_j)^2,
    # i.e. a quadratic form in z whose sector-level weighted aggregates are
    # penalized (squared) against a per-sector target X_star.
    sector = np.asarray(sector)
    X = np.asarray(X, dtype=float)
    sectors = np.unique(sector)
    n = len(X)
    M = np.zeros((len(sectors), n))
    for j, s in enumerate(sectors):
        mask = sector == s
        M[j, mask] = X[mask]
    Q = M.T @ M
    R = M.T @ np.asarray(X_star, dtype=float)
    c = 0.5 * np.sum(np.asarray(X_star, dtype=float) ** 2)
    return Q, R, c


def quadratic_form_risk(sector, X, X_star, z):
    # Reconstructed from usage (qf2.m/qf3.m/qf4.m): builds the (Q,R,c) triple
    # above and evaluates it at z, returning (qf, Q, R, c) matching the
    # MATLAB call's [qf,Q,R,c,results] signature (the diagnostic "results"
    # struct is not reproduced -- only what downstream scripts consume).
    Q, R, c = _sector_risk_matrices(sector, X, X_star)
    return quadratic_form(z, Q, R, c), Q, R, c


def _shift_by_benchmark(Q, R, c, b):
    # quadratic_form(x-b,Q,R,c) == quadratic_form(x,Q,R+Qb,0.5 b'Qb + b'R + c)
    # (qf1.m's own qf5a/qf5b identity) -- re-expresses a quadratic form in the
    # benchmark-relative deviation (w-b) as an equivalent one directly in w.
    b = np.asarray(b, dtype=float)
    return Q, R + Q @ b, 0.5 * b @ Q @ b + b @ R + c


def quadratic_form_bond_portfolio1(sector, varphi_MD, MD, MD_star, varphi_DTS, DTS, DTS_star,
                                    gamma_carry, carry):
    # Reconstructed from qf2.m: the absolute-target (not benchmark-relative)
    # mixed MD/DTS objective plus a linear carry term, evaluated directly on w.
    Q_MD, R_MD, c_MD = _sector_risk_matrices(sector, MD, MD_star)
    Q_DTS, R_DTS, c_DTS = _sector_risk_matrices(sector, DTS, DTS_star)
    Q = varphi_MD * Q_MD + varphi_DTS * Q_DTS
    R = varphi_MD * R_MD + varphi_DTS * R_DTS + gamma_carry * np.asarray(carry, dtype=float)
    c = varphi_MD * c_MD + varphi_DTS * c_DTS
    return Q, R, c


def quadratic_form_bond_portfolio2(sector, varphi_AS, varphi_MD, MD, MD_star, varphi_DTS, DTS,
                                    DTS_star, gamma_carry, carry, b):
    # Reconstructed from qf3.m/qf4.m: the benchmark-relative mixed AS/MD/DTS
    # objective (11a's basic3/basic4 R_Mix, generalized to an arbitrary
    # per-sector target rather than implicitly zero) plus a linear carry
    # term, re-expressed directly in w via the benchmark shift above. An
    # empty/None MD_star or DTS_star defaults to 0 -- "track the benchmark's
    # own sector composition exactly" -- which is the qf4.m/basic3 case;
    # qf3.m passes an explicit non-benchmark target instead.
    n = len(b)
    MD_star = np.zeros(len(np.unique(sector))) if MD_star is None else MD_star
    DTS_star = np.zeros(len(np.unique(sector))) if DTS_star is None else DTS_star

    Q_AS, R_AS, c_AS = _sector_risk_matrices(np.arange(n), np.ones(n), np.zeros(n))
    Q_MD, R_MD, c_MD = _sector_risk_matrices(sector, MD, MD_star)
    Q_DTS, R_DTS, c_DTS = _sector_risk_matrices(sector, DTS, DTS_star)

    Q_AS_b, R_AS_b, c_AS_b = _shift_by_benchmark(Q_AS, R_AS, c_AS, b)
    Q_MD_b, R_MD_b, c_MD_b = _shift_by_benchmark(Q_MD, R_MD, c_MD, b)
    Q_DTS_b, R_DTS_b, c_DTS_b = _shift_by_benchmark(Q_DTS, R_DTS, c_DTS, b)

    carry = np.asarray(carry, dtype=float)
    # The manual formula's carry term is linear in (w-b), i.e.
    # -gamma_carry*(w-b)@carry = -gamma_carry*w@carry + gamma_carry*b@carry --
    # shifting it into a term evaluated directly on w (matching quadratic_form's
    # -x@R convention) adds +gamma_carry*carry to R and +gamma_carry*(b@carry) to c.
    Q_b = varphi_AS * Q_AS_b + varphi_MD * Q_MD_b + varphi_DTS * Q_DTS_b
    R_b = (varphi_AS * R_AS_b + varphi_MD * R_MD_b + varphi_DTS * R_DTS_b
           + gamma_carry * carry)
    c_b = (varphi_AS * c_AS_b + varphi_MD * c_MD_b + varphi_DTS * c_DTS_b
           + gamma_carry * (np.asarray(b, dtype=float) @ carry))

    results = {"Q_AS_b": Q_AS_b, "R_AS_b": R_AS_b, "c_AS_b": c_AS_b,
               "Q_MD_b": Q_MD_b, "R_MD_b": R_MD_b, "c_MD_b": c_MD_b,
               "Q_DTS_b": Q_DTS_b, "R_DTS_b": R_DTS_b, "c_DTS_b": c_DTS_b}
    return Q_b, R_b, c_b, results


def bond_portfolio_metrics(sector, MD, DTS, b):
    # Reconstructed from qf4.m: portfolio-level and per-sector weighted MD/DTS.
    sector = np.asarray(sector)
    MD, DTS, b = np.asarray(MD, dtype=float), np.asarray(DTS, dtype=float), np.asarray(b, dtype=float)
    sectors = np.unique(sector)
    MD_b = np.sum(b * MD)
    DTS_b = np.sum(b * DTS)
    MD_j = np.array([np.sum((b * MD)[sector == s]) for s in sectors])
    DTS_j = np.array([np.sum((b * DTS)[sector == s]) for s in sectors])
    return MD_b, DTS_b, MD_j, DTS_j


## 1. `quadratic_form` — 14 algebraic identities

From `chap11_qf1.m`. Random $10\times10$ SPD matrices $Q_1=A_1'A_1$,
$Q_2=A_2'A_2$ and random vectors/scalars, checking linearity in
$(Q,R,c)$, positive scaling, the benchmark-shift identity used throughout
this chapter (`_shift_by_benchmark` above), and several structured
special cases (diagonal $Q$, rank-1 $Q$, and both restricted to a random
boolean mask $\omega$) against their closed-form scalar equivalents.

In [2]:
rng = np.random.default_rng(11)
n = 10
A1 = rng.standard_normal((n, n)); Q1 = A1.T @ A1
R1 = rng.standard_normal(n); c1 = rng.random()
A2 = rng.standard_normal((n, n)); Q2 = A2.T @ A2
R2 = rng.standard_normal(n); c2 = rng.random()

x = 2 * rng.random(n) - 1
y = 2 * rng.random(n) - 1

qf1 = quadratic_form(x, Q1, R1, c1)
qf2 = quadratic_form(x, Q2, R2, c2)
Q, R, c = Q1 + Q2, R1 + R2, c1 + c2
qf = quadratic_form(x, Q, R, c)
varphi = 1.3
qf3 = quadratic_form(x, varphi * Q, varphi * R, varphi * c)

qf4a, qf4b = qf, qf1 + qf2
qf5a = quadratic_form(x - y, Q, R, c)
qf5b = quadratic_form(x, Q, R + Q @ y, 0.5 * y @ Q @ y + y @ R + c)
qf6a = quadratic_form(x - y, Q, R, c)
qf6b = quadratic_form(y, Q, Q @ x - R, 0.5 * x @ Q @ x - x @ R + c)

q = np.diag(Q)
D_q = np.diag(q)
T_q = np.outer(q, q)
qf7a = 0.5 * np.sum(q * x * x)
qf7b = quadratic_form(x, D_q, np.zeros(n), 0)
qf8a = 0.5 * np.sum(q * (x - y) * (x - y))
qf8b = quadratic_form(x, D_q, D_q @ y, 0.5 * y @ D_q @ y)
qf9a = 0.5 * (np.sum(q * x) ** 2)
qf9b = quadratic_form(x, T_q, np.zeros(n), 0)
qf10a = 0.5 * (np.sum(q * (x - y)) ** 2)
qf10b = quadratic_form(x, T_q, T_q @ y, 0.5 * y @ T_q @ y)

omega = rng.random(n) < 0.5
omega_q = omega * q
D_omega_q = np.diag(omega_q)
T_omega_q = np.outer(omega_q, omega_q)

z = q * x * x
qf11a = 0.5 * z[omega].sum()
qf11b = quadratic_form(x, D_omega_q, np.zeros(n), 0)
z = q * (x - y) * (x - y)
qf12a = 0.5 * z[omega].sum()
qf12b = quadratic_form(x, D_omega_q, D_omega_q @ y, 0.5 * y @ D_omega_q @ y)
z = q * x
qf13a = 0.5 * (z[omega].sum() ** 2)
qf13b = quadratic_form(x, T_omega_q, np.zeros(n), 0)
z = q * (x - y)
qf14a = 0.5 * (z[omega].sum() ** 2)
qf14b = quadratic_form(x, T_omega_q, T_omega_q @ y, 0.5 * y @ T_omega_q @ y)

rows = [
    ("qf1 vs qf2 (independent, not an identity -- shown for reference)", qf1, qf2),
    ("qf3 = varphi*(qf1+qf2)", qf3, varphi * qf),
    ("qf4: additivity in (Q,R,c)", qf4a, qf4b),
    ("qf5: shift identity (x-y form)", qf5a, qf5b),
    ("qf6: shift identity (swap x<->y)", qf6a, qf6b),
    ("qf7: diagonal Q, x only", qf7a, qf7b),
    ("qf8: diagonal Q, x-y", qf8a, qf8b),
    ("qf9: rank-1 Q, x only", qf9a, qf9b),
    ("qf10: rank-1 Q, x-y", qf10a, qf10b),
    ("qf11: diagonal Q restricted to omega, x only", qf11a, qf11b),
    ("qf12: diagonal Q restricted to omega, x-y", qf12a, qf12b),
    ("qf13: rank-1 Q restricted to omega, x only", qf13a, qf13b),
    ("qf14: rank-1 Q restricted to omega, x-y", qf14a, qf14b),
]
table = pd.DataFrame(rows, columns=["Identity", "LHS", "RHS"])
table["diff"] = table["RHS"] - table["LHS"]
table.loc[0, "diff"] = np.nan  # qf1 vs qf2 isn't an identity, just reference values
display(table.round(10))

,Identity,LHS,RHS,diff
0,"qf1 vs qf2 (independent, not an identity -- sh...",15.383101,23.374242,NaN
1,qf3 = varphi*(qf1+qf2),50.384545,50.384545,-0.0
2,"qf4: additivity in (Q,R,c)",38.757342,38.757342,0.0
3,qf5: shift identity (x-y form),55.905629,55.905629,-0.0
4,qf6: shift identity (swap x<->y),55.905629,55.905629,-0.0
5,"qf7: diagonal Q, x only",32.156011,32.156011,0.0
6,"qf8: diagonal Q, x-y",47.550092,47.550092,-0.0
7,"qf9: rank-1 Q, x only",69.698949,69.698949,0.0
8,"qf10: rank-1 Q, x-y",1813.902880,1813.902880,-0.0
9,"qf11: diagonal Q restricted to omega, x only",12.560927,12.560927,-0.0


## 2. `quadratic_form_risk` / `quadratic_form_bond_portfolio1`

From `chap11_qf2.m`. A random long-only portfolio $w$ on an 8-asset/3-
sector universe; checks the reconstructed sector-level MD/DTS risk
functions (absolute per-sector targets `MD_star`/`DTS_star`, evaluated
directly on $w$, no carry) against a manual sector-loop computation.

In [3]:
b8 = np.array([22, 19, 17, 13, 11, 8, 6, 4]) / 100
MD8 = np.array([3.56, 7.48, 6.54, 10.23, 2.40, 2.30, 9.12, 7.96])
DTS8 = np.array([103, 155, 75, 796, 89, 45, 320, 245])
sector8 = np.array([1, 2, 1, 1, 2, 1, 2, 3])
carry8 = np.array([30, 30, 15, 60, 50, 25, 40, 38])
MD_star2 = np.array([5.00, 7.00, 7.00])
DTS_star2 = np.array([253.00, 115.00, 89.00])
varphi_MD2, varphi_DTS2, gamma_carry2 = 1, 0.01, 0.0

rng = np.random.default_rng(22)
w8 = rng.random(len(b8)); w8 = w8 / w8.sum()

sectors8 = np.unique(sector8)
qf1_MD = sum(0.5 * ((w8 * MD8)[sector8 == s].sum() - MD_star2[j]) ** 2 for j, s in enumerate(sectors8))
qf1_DTS = sum(0.5 * ((w8 * DTS8)[sector8 == s].sum() - DTS_star2[j]) ** 2 for j, s in enumerate(sectors8))
qf1 = varphi_MD2 * qf1_MD + varphi_DTS2 * qf1_DTS - gamma_carry2 * (w8 * carry8).sum()

qf2_MD, Q_MD, R_MD, c_MD = quadratic_form_risk(sector8, MD8, MD_star2, w8)
qf2_DTS, Q_DTS, R_DTS, c_DTS = quadratic_form_risk(sector8, DTS8, DTS_star2, w8)
Q, R, c = quadratic_form_bond_portfolio1(sector8, varphi_MD2, MD8, MD_star2,
                                          varphi_DTS2, DTS8, DTS_star2, gamma_carry2, carry8)
qf2 = quadratic_form(w8, Q, R, c)

table2 = pd.DataFrame({
    "manual (qf1)": [qf1_MD, qf1_DTS, qf1],
    "reconstructed (qf2)": [qf2_MD, qf2_DTS, qf2],
}, index=["MD term", "DTS term", "combined (varphi_MD*MD + varphi_DTS*DTS - gamma*carry)"])
table2["diff"] = table2["reconstructed (qf2)"] - table2["manual (qf1)"]
display(table2.round(10))

,manual (qf1),reconstructed (qf2),diff
MD term,29.303250,29.303250,0.0
DTS term,8347.427242,8347.427242,0.0
combined (varphi_MD*MD + varphi_DTS*DTS - gamma*carry),112.777522,112.777522,-0.0


## 3. `quadratic_form_bond_portfolio2` — explicit non-benchmark targets

From `chap11_qf3.m`. Same universe, now with the full AS+MD+DTS
benchmark-relative objective (evaluated on $w-b$) plus a carry term,
cross-checked three ways: a manual sector loop, the direct
`quadratic_form_risk` calls evaluated at $w-b$, and the benchmark-shifted
`(Q_b,R_b,c_b)` from `quadratic_form_bond_portfolio2` evaluated directly
at $w$ — all three must agree exactly.

In [4]:
varphi_AS3, varphi_MD3, varphi_DTS3, gamma_carry3 = 1, 1, 0.01, 0.50

rng = np.random.default_rng(33)
w8b = rng.random(len(b8)); w8b = w8b / w8b.sum()

d = w8b - b8
qf1_AS = 0.5 * np.sum(d ** 2)
qf1_MD = sum(0.5 * ((d * MD8)[sector8 == s].sum() - MD_star2[j]) ** 2 for j, s in enumerate(sectors8))
qf1_DTS = sum(0.5 * ((d * DTS8)[sector8 == s].sum() - DTS_star2[j]) ** 2 for j, s in enumerate(sectors8))
qf1_3 = varphi_AS3 * qf1_AS + varphi_MD3 * qf1_MD + varphi_DTS3 * qf1_DTS - gamma_carry3 * (d * carry8).sum()

n8 = len(b8)
qf2_AS, Q_AS, R_AS, c_AS = quadratic_form_risk(np.arange(n8), np.ones(n8), np.zeros(n8), d)
qf2_MD, Q_MD3, R_MD3, c_MD3 = quadratic_form_risk(sector8, MD8, MD_star2, d)
qf2_DTS, Q_DTS3, R_DTS3, c_DTS3 = quadratic_form_risk(sector8, DTS8, DTS_star2, d)
Q_b, R_b, c_b, res3 = quadratic_form_bond_portfolio2(sector8, varphi_AS3, varphi_MD3, MD8, MD_star2,
                                                      varphi_DTS3, DTS8, DTS_star2, gamma_carry3, carry8, b8)
qf2_3 = (varphi_AS3 * qf2_AS + varphi_MD3 * qf2_MD + varphi_DTS3 * qf2_DTS
         - gamma_carry3 * (d * carry8).sum())

qf3_AS = quadratic_form(w8b, res3["Q_AS_b"], res3["R_AS_b"], res3["c_AS_b"])
qf3_MD = quadratic_form(w8b, res3["Q_MD_b"], res3["R_MD_b"], res3["c_MD_b"])
qf3_DTS = quadratic_form(w8b, res3["Q_DTS_b"], res3["R_DTS_b"], res3["c_DTS_b"])
qf3_3 = quadratic_form(w8b, Q_b, R_b, c_b)

table3 = pd.DataFrame({
    "manual, on (w-b)": [qf1_AS, qf1_MD, qf1_DTS, qf1_3],
    "quadratic_form_risk, on (w-b)": [qf2_AS, qf2_MD, qf2_DTS, qf2_3],
    "portfolio2, benchmark-shifted, on w": [qf3_AS, qf3_MD, qf3_DTS, qf3_3],
}, index=["AS term", "MD term", "DTS term", "combined"])
table3["max diff"] = table3.max(axis=1) - table3.min(axis=1)
display(table3.round(10))

,"manual, on (w-b)","quadratic_form_risk, on (w-b)","portfolio2, benchmark-shifted, on w",max diff
AS term,0.022166,0.022166,0.022166,0.0
MD term,58.130125,58.130125,58.130125,0.0
DTS term,57444.019348,57444.019348,57444.019348,0.0
combined,632.830246,632.830246,632.830246,0.0


## 4. `bond_portfolio_metrics` and the default (benchmark-tracking) target

From `chap11_qf4.m`. A 5-asset/2-sector toy example: `bond_portfolio_metrics`
computes the benchmark's own portfolio- and sector-level MD/DTS, which
become the default target when `quadratic_form_bond_portfolio2` is
called with `MD_star=None`/`DTS_star=None` -- the benchmark-tracking case
used throughout `11a`'s `basic3`/`basic4` (targeting zero deviation from
the benchmark's own sector composition). Cross-checked against the same
manual sector-loop formula as Sections 2-3, now evaluated at three points
($w$, the benchmark $b$ itself, and the all-zero vector) to also confirm
$R_{\mathrm{Mix}}(b)=0$ (holding the benchmark exactly means zero
active-share/MD/DTS deviation *and* zero carry adjustment, since the
carry term is linear in $w-b$) and that $R_{\mathrm{Mix}}(0)$ matches the
function's own closed-form constant term $c_b$.

In [5]:
b5 = np.array([0.30, 0.25, 0.20, 0.15, 0.10])
w5 = np.array([0.20, 0.20, 0.20, 0.20, 0.20])
MD5 = np.array([3, 7, 6, 12, 2])
DTS5 = np.array([100, 250, 70, 400, 150])
sector5 = np.array([1, 2, 1, 1, 2])
carry5 = np.array([200, 300, 150, 250, 600])

MD_b, DTS_b, MD_j, DTS_j = bond_portfolio_metrics(sector5, MD5, DTS5, b5)
display(pd.DataFrame({"MD_j (per sector)": MD_j, "DTS_j (per sector)": DTS_j}))
print(f"Portfolio-level: MD_b = {MD_b:.2f}, DTS_b = {DTS_b:.2f}")

varphi_AS4, varphi_MD4, varphi_DTS4, gamma_carry4 = 1, 1, 0.001, 0.20
sectors5 = np.unique(sector5)

def manual_R_Mix(w, b, gamma):
    d = w - b
    AS = 0.5 * np.sum(d ** 2)
    MD_term = sum(0.5 * ((d * MD5)[sector5 == s].sum()) ** 2 for s in sectors5)
    DTS_term = sum(0.5 * ((d * DTS5)[sector5 == s].sum()) ** 2 for s in sectors5)
    return varphi_AS4 * AS + varphi_MD4 * MD_term + varphi_DTS4 * DTS_term - gamma * (d @ carry5)

Q_b5, R_b5, c_b5, _ = quadratic_form_bond_portfolio2(sector5, varphi_AS4, varphi_MD4, MD5, None,
                                                       varphi_DTS4, DTS5, None, gamma_carry4, carry5, b5)

points = {"w": w5, "b (benchmark itself)": b5, "0 (fully divested)": np.zeros(len(b5))}
rows = {name: (manual_R_Mix(pt, b5, gamma_carry4), quadratic_form(pt, Q_b5, R_b5, c_b5))
        for name, pt in points.items()}
table4 = pd.DataFrame(rows, index=["manual R_Mix", "quadratic_form(., Q_b, R_b, c_b)"]).T
table4["diff"] = table4["quadratic_form(., Q_b, R_b, c_b)"] - table4["manual R_Mix"]
display(table4.round(10))
print(f"c_b (closed-form constant term) = {c_b5:.6f}  vs.  R_Mix(0) = {rows['0 (fully divested)'][1]:.6f}")

,MD_j (per sector),DTS_j (per sector)
0,3.90,104.0
1,1.95,77.5


Portfolio-level: MD_b = 5.85, DTS_b = 181.50


,manual R_Mix,"quadratic_form(., Q_b, R_b, c_b)",diff
w,-7.378125,-7.378125,0.0
b (benchmark itself),0.000000,0.000000,0.0
0 (fully divested),70.529875,70.529875,0.0


c_b (closed-form constant term) = 70.529875  vs.  R_Mix(0) = 70.529875
